In [ ]:
from langchain.document_loaders import PyPDFium2Loader, PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

c:\Users\kiria\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Função para extrair (carregar) arquivos PDF de um diretório
def load_pdf_files(data):
    
    # Cria um loader (carregador) de documentos a partir de um diretório
    loader = DirectoryLoader(
        
        data,  # Caminho da pasta onde estão os arquivos PDF
        
        glob="*.pdf",  # Define que ele deve pegar apenas arquivos com extensão .pdf
        
        loader_cls=PyPDFium2Loader  # Define qual classe será usada para ler os PDFs
                                     # (nesse caso, PyPDFium2Loader, que extrai o texto dos PDFs)
    )

    documents = loader.load()
    return documents

In [5]:
extracted_data = load_pdf_files(
    "C:/Users/kiria/OneDrive/Documentos/LLM/A-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS/data"
)

In [6]:
len(extracted_data)

4505

In [7]:
from typing import List 
from langchain.schema import Document

#essa função serve para filtrar as informações, o python trouxe várias coisas, como autor do livro, data de publicação e essas informações são irrelevantes.
#então eu quero apenas o source(fonte) e a page content(conteúdo da página)
def filter_to_minimal_docs(docs: list[Document]) -> List[Document]:
    #given a list of document objects, return a new list of document objects containing only 'source' in metadata and the original page_content
    minimal_docs: list[Document] = []


    minimal_docs: list[Document]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
                )
            )
    return minimal_docs

In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
# split the document into smaller chunks 
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20, 
        )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [10]:
text_chunk = text_split(minimal_docs)
print(f"number of chunks:  {len(text_chunk)}")

number of chunks:  40257


In [14]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embedding():
    #download and return the Hugging face embeddings model.

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings


embedding = download_embedding()

C:\Users\kiria\AppData\Local\Temp\ipykernel_9056\936082699.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\kiria\anaconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kiria\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` enviro

In [15]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vector = embedding.embed_query("hello word")
print(vector)

#isso aqui é o modelo de embeddings transformando um texto cru em um vetor de números que representa essa frase

[-0.02205732837319374, 0.03135434165596962, 0.04234997183084488, 0.026110919192433357, -0.06519321352243423, -0.07004162669181824, 0.05430905148386955, -0.0016129486029967666, -0.06259442120790482, 0.03530611842870712, 0.02730604261159897, 0.051380790770053864, 0.09750933945178986, -0.06328085064888, 0.008590356446802616, 0.02296951599419117, 0.0637909322977066, -0.02689855359494686, -0.13075025379657745, 0.03945804759860039, -0.04759364575147629, 0.051643915474414825, -0.008419344201683998, 0.0008068453171290457, -0.02593022584915161, -0.011951646767556667, -0.015885207802057266, 0.015778249129652977, 0.04706128314137459, -0.07030077278614044, 0.050214845687150955, 0.020775917917490005, 0.13571418821811676, 0.0236296895891428, 0.01985790953040123, 0.03593345358967781, -0.046726323664188385, -0.062266506254673004, -0.013844909146428108, -0.012370150536298752, -0.03176111355423927, -0.0936770886182785, -0.034933894872665405, -0.025743627920746803, 0.07390404492616653, -0.053186863660812

In [19]:
print(f"vector lenght: {len(vector)}")

vector lenght: 384


In [21]:
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [24]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [26]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
from pinecone import ServerlessSpec

index_name= "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of embedding
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    
index = pc.Index(index_name)

#if the vector dimension is high, that means there are more information in the vector

#higher the vector dimension, embedding model will be better

In [35]:
#store our vector -> pinecone

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
    )

In [36]:
#Load existing index

from langchain_pinecone import PineconeVectorStore

#embed each chunk and upsert the embeddings into your pinecone index

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
    )

# Add more data to the existing Pinecone Index